# Domain Expansion Detector — Hand Gesture Classifier

In [ ]:
import os
import glob
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt

DATASET_DIR = "dataset"
CLASS_NAMES = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])
CLASS_NAMES

## Load image

In [ ]:
def load_image(path, max_side=512):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    h, w = img.shape[:2]
    scale = max_side / float(max(h, w))
    if scale < 1.0:
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return img

## Color segmentation

Combine HSV and YCrCb thresholds — each catches skin tones the other misses, and intersecting them cuts false positives from warm backgrounds.

In [ ]:
def skin_mask(bgr):
    blurred = cv2.GaussianBlur(bgr, (5, 5), 0)

    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    hsv_mask = cv2.inRange(hsv, (0, 30, 60), (20, 150, 255))

    ycrcb = cv2.cvtColor(blurred, cv2.COLOR_BGR2YCrCb)
    ycrcb_mask = cv2.inRange(ycrcb, (0, 135, 85), (255, 180, 135))

    return cv2.bitwise_and(hsv_mask, ycrcb_mask)

## Morphological cleanup

Close holes inside the hand, open to drop speckle, then keep only the largest connected component — this removes stray skin-colored pixels in the background without needing a reference frame.

In [ ]:
def clean_mask(mask):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.GaussianBlur(mask, (5, 5), 0)
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num <= 1:
        return mask
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return np.where(labels == largest, 255, 0).astype(np.uint8)

## Canny edge detection

In [ ]:
def canny_edges(mask):
    return cv2.Canny(mask, 100, 200)

## Contour extraction

In [ ]:
def largest_contour(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    return max(contours, key=cv2.contourArea)

## Hu moments

In [ ]:
def hu_moments(contour):
    pass

## Combine to Detect Gesture

In [ ]:
def process(path):
    bgr = load_image(path)
    mask = skin_mask(bgr)
    mask = clean_mask(mask)
    edges = canny_edges(mask)
    contour = largest_contour(mask)
    if contour is None:
        return {"bgr": bgr, "mask": mask, "edges": edges, "contour": None, "hu": None}
    return {
        "bgr": bgr,
        "mask": mask,
        "edges": edges,
        "contour": contour,
        "hu": hu_moments(contour),
    }

## 7. Build reference dataset & classify

For each class folder, compute contours for every image and store them. At query time, use `cv2.matchShapes` (which uses log-Hu moments internally, method I2) against every reference and take the class of the nearest match.